In [123]:
%load_ext autoreload
%autoreload 2

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["OMP_NUM_THREADS"] = "1"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [124]:
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

from modelutils import *
from datautils import *

In [125]:
def get_opt(model):
    import torch
    def skip(*args, **kwargs):
        pass
    torch.nn.init.kaiming_uniform_ = skip
    torch.nn.init.uniform_ = skip
    torch.nn.init.normal_ = skip
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(model, torch_dtype='auto', cache_dir="/home/jmuravska/diplomka/smollm/llm_weights")
    print("ms", model.config.max_position_embeddings)
    # This override is needed for Llama3-8B (otherwise seqlen would be huge)
    model.seqlen = model.config.max_position_embeddings #1024*8
    return model

In [126]:
# Create model

model_name = "HuggingFaceTB/SmolLM2-135M"

model = get_opt(model_name)
model.eval()

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

ms 8192


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-05)
    (rotary_emb): Lla

In [127]:
n_samples = 32

dataloader, testloader = get_loaders(
    "c4", nsamples=n_samples, seed=0, model=model_name, seqlen=model.seqlen
)

tok 0 0


Token indices sequence length is longer than the specified maximum sequence length for this model (8597 > 8192). Running this sequence through the model will result in indexing errors


done 1
done 2
done 3
done 4
done 5
done 6
done 7
done 8
done 9
done 10
done 11
done 12
done 13
done 14
done 15
done 16
done 17
done 18
done 19
done 20
done 21
done 22
done 23
done 24
done 25
done 26
done 27
done 28
done 29
done 30
done 31
done 32


In [128]:
sparsity = 0.5
MAX_ITER = 100
RANDOM_SWAPS = 100
BLOCK_SIZE = (1, 2)

def do_block_wanda_sparsity_per_row(lx):
    input_sq_norms = lx.input_sq_norms
    norm = input_sq_norms.sqrt() + 1e-8
    
    scores = (lx.weight.float().detach() * norm).abs()
    rows, cols = scores.shape
    block_rows, block_cols = BLOCK_SIZE

    n_blocks_row = rows // block_rows
    n_blocks_col = cols // block_cols

    cropped_scores = scores[:n_blocks_row*block_rows, :n_blocks_col*block_cols]
    blocks = cropped_scores.reshape(n_blocks_row, block_rows, n_blocks_col, block_cols)

    # Shape: (n_blocks_row, n_blocks_col)
    block_norms = blocks.sum(dim=(1, 3))

    # Find the exact column index to cut off the bottom 50%
    k = int(n_blocks_col * sparsity)
    
    # Sort each row individually, and grab the threshold for THAT specific row
    threshold = block_norms.sort(dim=1)[0][:, k]

    # Create block mask (compare each block to its row's specific threshold)
    block_mask = block_norms >= threshold.unsqueeze(1)

    # Expand block mask to original size
    mask = block_mask.repeat_interleave(block_rows, dim=0).repeat_interleave(block_cols, dim=1)

    # Handle padding for remainders
    pad_bottom = rows - mask.shape[0]
    pad_right = cols - mask.shape[1]
    
    if pad_bottom > 0 or pad_right > 0:
        mask = F.pad(mask, (0, pad_right, 0, pad_bottom), value=False)

    mask = mask.to(lx.weight.device)
    W_pruned = lx.weight * mask

    return W_pruned


import numpy as np
import sys
sys.path.append(os.path.abspath('..'))
from tetris import tetris_pruning, block_sparsity_pruning


def do_tetris_block_wanda(lx):
    # 1. Extract pytorch weights and calculate Wanda Scores in numpy
    W_pt = lx.weight.detach()
    device = W_pt.device
    dtype = W_pt.dtype
    
    W_np = W_pt.float().cpu().numpy()
    input_sq_norms_np = lx.input_sq_norms.float().cpu().numpy()
    
    norm = np.sqrt(input_sq_norms_np) + 1e-8
    wanda_scores = np.abs(W_np) * norm
    
    # 2. Run Tetris on the Wanda Scores
    _, _, perm, _, _, _ = tetris_pruning(
        W=wanda_scores, 
        block_size=BLOCK_SIZE, 
        sparsity=sparsity,
        max_iter=MAX_ITER,
        random_swaps=RANDOM_SWAPS,
        verbose=False
    )

    permuted_scores = wanda_scores[:, perm]
    _, best_score_mask = block_sparsity_pruning(permuted_scores, block_size=BLOCK_SIZE, sparsity=sparsity)
    
    # 3. Apply permutation and mask
    W_permuted = W_np[:, perm]
    W_pruned_np = W_permuted * best_score_mask
    
    # 4. Un-permute back to the original input order for safe evaluation
    inverse_perm = np.argsort(perm)
    W_final_unpermuted = W_pruned_np[:, inverse_perm]
    
    # 5. Convert back to pytorch
    W_pruned_pt = torch.from_numpy(W_final_unpermuted).to(device=device, dtype=dtype)
    
    return W_pruned_pt



# def do_wanda(lx):
#     start = time.time()
#     input_sq_norms = lx.input_sq_norms
    
#     norm = input_sq_norms.sqrt() + 1e-8
#     # Actual score for pruning is weight times input norm
#     scores = (lx.weight.float().detach() * norm).abs()
    
#     # Each output gets same sparsity, but for our block pruning we keep global sparsity
#     thres = scores.sort(dim=1)[0][:,int(scores.shape[1] * sparsity)]
#     mask = scores >= thres.unsqueeze(1)
       
#     return lx.weight * mask

In [129]:
@torch.no_grad()
def opt_sequential(model, dataloader, dev):
    print('Starting ...')

    use_cache = model.config.use_cache
    model.config.use_cache = False
    layers = model.model.layers

    model.model.embed_tokens = model.model.embed_tokens.to(dev) 
    model.model.rotary_emb = model.model.rotary_emb.to(dev)
    layers[0] = layers[0].to(dev)

    dtype = next(iter(model.parameters())).dtype
    inps = torch.zeros(
        (n_samples, model.seqlen, model.config.hidden_size), dtype=dtype, device=dev
    )
    cache = {'i': 0, 'attention_mask': None}

    class Catcher(nn.Module):
        def __init__(self, module):
            super().__init__()
            self.module = module
        def forward(self, inp, **kwargs):
            inps[cache['i']] = inp
            cache['i'] += 1
            cache['attention_mask'] = kwargs['attention_mask']
            cache['position_embeddings'] = kwargs['position_embeddings']
            raise ValueError
    layers[0] = Catcher(layers[0])
    for batch in dataloader:
        try:
            model(batch[0].to(dev))
        except ValueError:
            pass
    layers[0] = layers[0].module

    layers[0] = layers[0].cpu()
    model.model.embed_tokens = model.model.embed_tokens.cpu()
    torch.cuda.empty_cache()

    outs = torch.zeros_like(inps)
    attention_mask = cache['attention_mask']
    position_embeddings = cache['position_embeddings']

    print('Ready.')

    old_subset = {}
    # Run pruning and collect input norm on-the-fly
    for i in range(len(layers)):
        layer = layers[i].to(dev)

        subset = find_layers(layer)

        def add_batch(name):
            def tmp(layer, inp, out):
                X = inp[0].detach().float()
                X = X.reshape(-1, X.shape[-1])
                layer.input_sq_norms += X.square().mean(dim=0)
            return tmp
        handles = []
        for name in subset:
            subset[name].batches = []
            subset[name].input_sq_norms = torch.zeros(subset[name].weight.shape[1], device=subset[name].weight.device)        
            handles.append(subset[name].register_forward_hook(add_batch(name)))
        for j in range(n_samples):
            outs[j] = layer(inps[j].unsqueeze(0), attention_mask=attention_mask, position_embeddings=position_embeddings)[0]
        for h in handles:
            h.remove()

        for name in subset:
            print(i, name)
            print('Pruning ...')
            lx = subset[name]
            
            s1 = time.time()
            W2 = do_block_wanda_sparsity_per_row(lx)
            
            print(W2.shape, lx.weight.shape)
            # TODO fix this error
            print("rel err", ((lx.weight - W2).float().square() * lx.input_sq_norms).sum().item() / (lx.weight.float().square() * lx.input_sq_norms).sum().item(),
                  "density", (W2 != 0).sum().item() / W2.numel())
            lx.weight.data = W2.to(lx.weight)
            

        for j in range(n_samples):
            outs[j] = layer(inps[j].unsqueeze(0), attention_mask=attention_mask, position_embeddings=position_embeddings)[0]
        
        

        layers[i] = layer.cpu()
        del layer
        
        
        
        torch.cuda.empty_cache()

        inps, outs = outs, inps
        old_subset = subset

    model.config.use_cache = use_cache

start = time.time()
opt_sequential(model, dataloader, DEV)
print("total time", time.time() - start)

Starting ...
Ready.
0 self_attn.q_proj
Pruning ...
torch.Size([576, 576]) torch.Size([576, 576])
rel err 0.005936160967720333 density 0.5
0 self_attn.k_proj
Pruning ...
torch.Size([192, 576]) torch.Size([192, 576])
rel err 0.008084999157725339 density 0.5
0 self_attn.v_proj
Pruning ...
torch.Size([192, 576]) torch.Size([192, 576])
rel err 0.0496797981030928 density 0.5
0 self_attn.o_proj
Pruning ...
torch.Size([576, 576]) torch.Size([576, 576])
rel err 0.0005978278200100358 density 0.5
0 mlp.gate_proj
Pruning ...
torch.Size([1536, 576]) torch.Size([1536, 576])
rel err 0.08195994042874735 density 0.5
0 mlp.up_proj
Pruning ...
torch.Size([1536, 576]) torch.Size([1536, 576])
rel err 0.13360896173756642 density 0.5
0 mlp.down_proj
Pruning ...
torch.Size([576, 1536]) torch.Size([576, 1536])
rel err 0.09691206233711824 density 0.5
1 self_attn.q_proj
Pruning ...
torch.Size([576, 576]) torch.Size([576, 576])
rel err 0.030320529344428717 density 0.5
1 self_attn.k_proj
Pruning ...
torch.Size([19

In [130]:
@torch.no_grad()
def opt_eval(model, testenc, dev, dataset: str, log_wandb: bool = False):
    print('Evaluating ...')

    testenc = testenc.input_ids
    nsamples = testenc.numel() // model.seqlen

    use_cache = model.config.use_cache
    model.config.use_cache = False
    layers = model.model.layers

    model.model.embed_tokens = model.model.embed_tokens.to(dev)
    model.model.rotary_emb = model.model.rotary_emb.to(dev)
    layers[0] = layers[0].to(dev)

    dtype = next(iter(model.parameters())).dtype
    inps = torch.zeros(
        (nsamples, model.seqlen, model.config.hidden_size), dtype=dtype, device=dev
    )
    cache = {'i': 0, 'attention_mask': None}

    class Catcher(nn.Module):
        def __init__(self, module):
            super().__init__()
            self.module = module
        def forward(self, inp, **kwargs):
            inps[cache['i']] = inp
            cache['i'] += 1
            cache['attention_mask'] = kwargs['attention_mask']
            cache['position_embeddings'] = kwargs['position_embeddings']
            raise ValueError
    layers[0] = Catcher(layers[0])
    for i in range(nsamples):
        batch = testenc[:, (i * model.seqlen):((i + 1) * model.seqlen)].to(dev)
        try:
            model(batch)
        except ValueError:
            pass
    layers[0] = layers[0].module

    layers[0] = layers[0].cpu()
    model.model.embed_tokens = model.model.embed_tokens.cpu()
    torch.cuda.empty_cache()

    outs = torch.zeros_like(inps)
    attention_mask = cache['attention_mask']
    position_embeddings = cache['position_embeddings']

    for i in range(len(layers)):
        print(i)
        layer = layers[i].to(dev)

        
        for j in range(nsamples):
            outs[j] = layer(inps[j].unsqueeze(0), attention_mask=attention_mask, position_embeddings=position_embeddings)[0]
        layers[i] = layer.cpu()
        del layer
        torch.cuda.empty_cache()
        inps, outs = outs, inps

    if model.model.norm is not None:
        model.model.norm = model.model.norm.to(dev)
    model.lm_head = model.lm_head.to(dev)

    testenc = testenc.to(dev)
    nlls = []
    for i in range(nsamples):
        hidden_states = inps[i].unsqueeze(0)
        if model.model.norm is not None:
            hidden_states = model.model.norm(hidden_states)
        lm_logits = model.lm_head(hidden_states)
        shift_logits = lm_logits[:, :-1, :].contiguous()
        shift_labels = testenc[
            :, (i * model.seqlen):((i + 1) * model.seqlen)
        ][:, 1:]
        loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        neg_log_likelihood = loss.float() * model.seqlen
        nlls.append(neg_log_likelihood)
    ppl = torch.exp(torch.stack(nlls).sum() / (nsamples * model.seqlen))
    print(f"Perplexity: {ppl.item():3f}")
    if log_wandb:
         wandb.log({f'{dataset}/perplexity': ppl.item()})

    model.config.use_cache = use_cache

In [131]:
for dataset in ['wikitext2']:
    dataloader, testloader = get_loaders(
        dataset, seed=0, model=model_name, seqlen=model.seqlen
    )
    print(dataset)
    opt_eval(model, testloader, DEV, dataset, False)

tok 0 0


Token indices sequence length is longer than the specified maximum sequence length for this model (2565489 > 8192). Running this sequence through the model will result in indexing errors


wikitext2
Evaluating ...
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
Perplexity: 304.266998
